In [1]:
import pandas as pd
import zipfile
from pathlib import Path

# נעלה את קובץ הZIP של נתוני GTFS 

In [2]:
gtfs_zip_path = Path(r"C:\Users\User\Documents\שנה ג\data_science\פרוייקט\israel-public-transportation.zip")


In [3]:
# =========================
# Step 1: Inspect GTFS ZIP content
# =========================

with zipfile.ZipFile(gtfs_zip_path, "r") as z:
    gtfs_files = z.namelist()

gtfs_files

['agency.txt',
 'calendar.txt',
 'fare_attributes.txt',
 'fare_rules.txt',
 'routes.txt',
 'shapes.txt',
 'stop_times.txt',
 'stops.txt',
 'translations.txt',
 'trips.txt']

# קריאת הנתונים החשובים - 

 routes.txt     -> מידע על הקווים: מספר קו, route_id, שם/תיאור הקו


 trips.txt      -> נסיעות ספציפיות ששייכות לכל route_id


 stop_times.txt -> סדר התחנות בכל נסיעה


 stops.txt      -> שמות התחנות, קוד תחנה ומיקום גיאוגרפי


 agency.txt     -> מידע על מפעילי התחבורה הציבורית

In [4]:
# =========================
# Step 2: Read GTFS files into DataFrames
# =========================

with zipfile.ZipFile(gtfs_zip_path, "r") as z:
    routes = pd.read_csv(z.open("routes.txt"))
    trips = pd.read_csv(z.open("trips.txt"), encoding="utf-8-sig")
    stop_times = pd.read_csv(z.open("stop_times.txt"), encoding="utf-8-sig")
    stops = pd.read_csv(z.open("stops.txt"), encoding="utf-8-sig")
    agency = pd.read_csv(z.open("agency.txt"), encoding="utf-8-sig")




In [5]:
# =========================
# Step 3: Inspect columns
# =========================


print("routes columns:")
print(routes.columns.tolist())

print("\ntrips columns:")
print(trips.columns.tolist())

print("\nstop_times columns:")
print(stop_times.columns.tolist())

print("\nstops columns:")
print(stops.columns.tolist())

print("\nagency columns:")
print(agency.columns.tolist())


routes columns:
['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_desc', 'route_type', 'route_color']

trips columns:
['route_id', 'service_id', 'trip_id', 'trip_headsign', 'direction_id', 'shape_id', 'wheelchair_accessible']

stop_times columns:
['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence', 'pickup_type', 'drop_off_type', 'shape_dist_traveled']

stops columns:
['stop_id', 'stop_code', 'stop_name', 'stop_desc', 'stop_lat', 'stop_lon', 'location_type', 'parent_station', 'zone_id']

agency columns:
['agency_id', 'agency_name', 'agency_url', 'agency_timezone', 'agency_lang', 'agency_phone', 'agency_fare_url']


# מציאת קווים ספציפים 

In [6]:
# ============================================================
# Function: Find routes by line number and area
# ============================================================

def find_routes_by_line_and_area(routes_df, line_number, area_keywords, agency_df=None):
    """
    Find GTFS routes by public line number and area/city keywords.

    Parameters
    ----------
    routes_df : pandas DataFrame
        GTFS routes.txt table.

    line_number : str or int
        Public bus line number, for example 15, 18, 77.

    area_keywords : str or list of str
        Area/city keywords to search in route text fields.
        Example:
        "ירושלים"
        or
        ["ירושלים", "Jerusalem"]

    agency_df : pandas DataFrame, optional
        GTFS agency.txt table. If provided, agency_name will be added.

    Returns
    -------
    pandas DataFrame
        Candidate routes matching both the line number and area keywords.
    """

    # -----------------------------
    # 1. Normalize inputs
    # -----------------------------

    line_number = str(line_number).strip()

    if isinstance(area_keywords, str):
        area_keywords = [area_keywords]

    area_keywords = [str(keyword).strip() for keyword in area_keywords]

    # -----------------------------
    # 2. Filter by line number
    # -----------------------------
    # route_short_name is usually the public bus line number.

    routes_line = routes_df[
        routes_df["route_short_name"].astype(str).str.strip() == line_number
    ].copy()

    if routes_line.empty:
        print(f"No routes found for line number: {line_number}")
        return routes_line

    # -----------------------------
    # 3. Add agency name if available
    # -----------------------------

    if agency_df is not None:
        if "agency_id" in routes_line.columns and "agency_id" in agency_df.columns:
            agency_cols = [
                col for col in [
                    "agency_id",
                    "agency_name",
                    "agency_url",
                    "agency_timezone"
                ]
                if col in agency_df.columns
            ]

            routes_line = routes_line.merge(
                agency_df[agency_cols].drop_duplicates(),
                on="agency_id",
                how="left"
            )

    # -----------------------------
    # 4. Decide which text columns to search
    # -----------------------------

    possible_text_cols = [
        "route_long_name",
        "route_desc",
        "agency_name"
    ]

    text_cols = [
        col for col in possible_text_cols
        if col in routes_line.columns
    ]

    if len(text_cols) == 0:
        print("No textual columns found for area filtering.")
        return routes_line

    # -----------------------------
    # 5. Filter by area keywords
    # -----------------------------

    area_mask = pd.Series(False, index=routes_line.index)

    for col in text_cols:
        col_text = routes_line[col].astype(str)

        for keyword in area_keywords:
            area_mask = area_mask | col_text.str.contains(
                keyword,
                case=False,
                na=False,
                regex=False
            )

    routes_line_area = routes_line[area_mask].copy()

    # -----------------------------
    # 6. Choose useful columns to display
    # -----------------------------

    cols_to_show = [
        col for col in [
            "route_id",
            "agency_id",
            "agency_name",
            "route_short_name",
            "route_long_name",
            "route_desc",
            "route_type"
        ]
        if col in routes_line_area.columns
    ]

    if routes_line_area.empty:
        print(
            f"No routes found for line {line_number} "
            f"with area keywords: {area_keywords}"
        )

        print("\nShowing all routes found for this line number instead:")
        cols_all = [
            col for col in [
                "route_id",
                "agency_id",
                "agency_name",
                "route_short_name",
                "route_long_name",
                "route_desc",
                "route_type"
            ]
            if col in routes_line.columns
        ]

        return routes_line[cols_all].drop_duplicates().reset_index(drop=True)

    return routes_line_area[cols_to_show].drop_duplicates().reset_index(drop=True)

In [7]:
selected_routes = [15,22,19,17,9,97,14]
routes_selected = pd.DataFrame()


for line in selected_routes:
    routes_line = find_routes_by_line_and_area(
        routes_df=routes,
        line_number=line,
        area_keywords=["ירושלים", "Jerusalem"],
        agency_df=agency
    )

    routes_selected = pd.concat([routes_selected, routes_line], ignore_index=True)


routes_selected


,route_id,agency_id,agency_name,route_short_name,route_long_name,route_desc,route_type
0,37936,3,אגד,15,תחנה תפעולית/חניון תלפיות-ירושלים<->האומן/ברעם...,59015-3-#,3
1,5499,3,אגד,22,תחנה תפעולית/חניון תלפיות-ירושלים<->מסוף 700/ש...,12022-1-#,3
2,5502,3,אגד,22,שדרות נווה יעקב/משה סנה-ירושלים<->חניון תלפיות...,12022-2-#,3
3,10802,3,אגד,19,כניסה ראשית/הדסה עין כרם-ירושלים<->מסוף הר הצו...,15019-1-א,3
4,10803,3,אגד,19,דולצ'ין/קוליץ-ירושלים<->מסוף הר הצופים/מרטין ב...,15019-1-ד,3
5,10804,3,אגד,19,מסוף הר הצופים/בנימין מזר-ירושלים<->כניסה ראשי...,15019-2-#,3
6,10806,3,אגד,19,מסוף גילה/המרגלית-ירושלים<->כניסה ראשית/הדסה ע...,15019-2-ב,3
7,10807,3,אגד,19,צומת פת/גולומב-ירושלים<->כניסה ראשית/הדסה עין ...,15019-2-ד,3
8,10398,3,אגד,17,אריה דולצ'ין/יעקב צור-ירושלים<->מסוף הר הצופים...,18017-1-#,3
9,10399,3,אגד,17,מסוף הר הצופים/בנימין מזר-ירושלים<->דולצ'ין/בן...,18017-2-#,3


מציאת ID לכל נסיעה 

In [8]:
routes_id_dict = {} 
for line in selected_routes:
    line_str = str(line).strip()
    route_id_for_line =(routes_selected.loc[routes_selected['route_short_name'].astype(str).str.strip() == line_str, 'route_id'].dropna().unique().tolist())
    routes_id_dict[line] = route_id_for_line
routes_id_dict

{15: [37936],
 22: [5499, 5502],
 19: [10802, 10803, 10804, 10806, 10807],
 17: [10398, 10399],
 9: [11107, 11108],
 97: [36950, 36951],
 14: [10179, 10180]}

צצמצום - שיהיה ID אחד לכל קו 

In [14]:
chosen_route_ids_by_line = {
    15: ["37936"],
    22: ["5499", "5502"],
    19: ["10802", "10803", "10804", "10806", "10807"],
    17: ["10398", "10399"],
     9: ["11107", "11108"],
    97: ["36950", "36951"],
    14: ["10179", "10180"]
}
# ============================================================
# Convert selected route_ids dictionary to a list
# ============================================================

chosen_route_ids = [
    str(route_id).strip()
    for route_id in chosen_route_ids_by_line.values()
]

chosen_route_ids

["['37936']",
 "['5499', '5502']",
 "['10802', '10803', '10804', '10806', '10807']",
 "['10398', '10399']",
 "['11107', '11108']",
 "['36950', '36951']",
 "['10179', '10180']"]

In [10]:
Unique_stop_codes_15= {1024, 1025, 1028, 6286, 1039, 1021, 6299, 6300, 1053, 2590, 2591, 1963, 1856, 9921, 1858, 5952, 463, 5200, 4050, 2258, 1620, 6229, 2260, 2261, 1016, 1754, 5979, 2399, 5987, 5988, 3300, 3301, 1255, 3304, 3302, 1258, 2540, 623, 1905, 1015, 1013, 1014, 1655, 4216, 1017, 6267, 6269, 1022}
# find names of stops


In [15]:
all_routes = r"C:\Users\User\Documents\שנה ג\data_science\data-science\govData\ride_data_merged.csv"
rides_df = pd.read_csv(all_routes, encoding="utf-8-sig")

unique_stop_codes_in_rides = set(rides_df["StopCode"].unique())
print(f"Unique stop codes in rides data: {len(unique_stop_codes_in_rides)}")

# make set from unique stop codes in rides data
 
Unique_stop_codes = unique_stop_codes_in_rides

Unique stop codes in rides data: 446


In [16]:
# ============================================================
# Function: Find stops by stop codes and area keywords
# ============================================================

def find_stops_by_codes_and_area(stops_df, stop_codes_set, area_keywords):
    """
    Find GTFS stops by a specific set of stop codes and area/city keywords.

    Parameters
    ----------
    stops_df : pandas DataFrame
        GTFS stops.txt table.

    stop_codes_set : set or list
        A collection of unique stop codes (e.g., Unique_stop_codes_15).

    area_keywords : str or list of str
        Area/city keywords to search in stop text fields.
        Example: ["ירושלים", "Jerusalem"]

    Returns
    -------
    pandas DataFrame
        Stops matching both the stop codes and area keywords.
    """

    # -----------------------------
    # 1. Normalize inputs
    # -----------------------------
    if isinstance(area_keywords, str):
        area_keywords = [area_keywords]
    
    area_keywords = [str(keyword).strip() for keyword in area_keywords]
    
    # מבטיחים שכל הקודים בסט נבדקים כמספרים שלמים (או כטקסט, תלוי בטיפוס בטבלה)
    stop_codes_clean = [int(code) for code in stop_codes_set]

    # -----------------------------
    # 2. Filter by stop codes
    # -----------------------------
    # מסננים קודם כל לפי קודי התחנות שנתת
    stops_filtered = stops_df[
        stops_df["stop_code"].astype(int).isin(stop_codes_clean)
    ].copy()

    if stops_filtered.empty:
        print("No stops found matching the provided stop codes.")
        return stops_filtered

    # -----------------------------
    # 3. Decide which text columns to search
    # -----------------------------
    # נחפש בעמודות השם והתיאור של התחנה
    possible_text_cols = ["stop_name", "stop_desc"]
    text_cols = [col for col in possible_text_cols if col in stops_filtered.columns]

    if len(text_cols) == 0:
        print("No textual columns found for area filtering.")
        return stops_filtered

    # -----------------------------
    # 4. Filter by area keywords
    # -----------------------------
    area_mask = pd.Series(False, index=stops_filtered.index)

    for col in text_cols:
        col_text = stops_filtered[col].astype(str)

        for keyword in area_keywords:
            area_mask = area_mask | col_text.str.contains(
                keyword,
                case=False,
                na=False,
                regex=False
            )

    stops_final = stops_filtered[area_mask].copy()

    # -----------------------------
    # 5. Choose useful columns to display
    # -----------------------------
    cols_to_show = [
        col for col in [
            "stop_id",
            "stop_code",
            "stop_name",
            "stop_desc",
            "stop_lat",
            "stop_lon"
        ]
        if col in stops_final.columns
    ]

    # אם הסינון של האזור החזיר ריק, נתנהג כמו בפונקציה המקורית ונציג את הכל עם התראה
    if stops_final.empty:
        print(f"No stops found with area keywords: {area_keywords}")
        print("\nShowing all stops found for these codes instead:")
        return stops_filtered[cols_to_show].drop_duplicates().reset_index(drop=True)

    return stops_final[cols_to_show].drop_duplicates().reset_index(drop=True)

In [17]:
# קריאה לפונקציה עם הנתונים שלך
jerusalem_stops_15 = find_stops_by_codes_and_area(
    stops_df=stops,
    stop_codes_set=Unique_stop_codes,
    area_keywords=["ירושלים", "Jerusalem"]
)

# הצגת התוצאה
jerusalem_stops_15
# save path 
output_path = r"C:\Users\User\Documents\שנה ג\data_science\data-science\govData\jerusalem_stops.csv"
# save to csv
jerusalem_stops_15.to_csv(output_path, index=False, encoding="utf-8-sig")


